In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, avg, max, min, round, month, year, concat, lit, lag, date_format, stddev

In [0]:
gold_df = (spark.read.table("lakehouse.silver.precos_combustiveis"))

In [0]:
preco_medio_mensais_por_bandeira_df = (
    spark.read.table("lakehouse.silver.precos_combustiveis")
    .withColumn("mes_ano_coleta", concat(year("data_da_coleta"), lit("-"), date_format("data_da_coleta", "MM")))
    .groupBy("bandeira", "regiao_sigla")
    .pivot("mes_ano_coleta")
    .agg(round(avg("valor_de_venda"), 3))
    .orderBy("bandeira", "regiao_sigla")
)

preco_medio_mensais_por_bandeira_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lakehouse.gold.preco_medio_mensais_por_bandeira")

In [0]:
preco_medio_max_min_estado_ano_df = (
    spark.read.table("lakehouse.silver.precos_combustiveis")
    .withColumn("mes_ano_coleta", concat(year("data_da_coleta"), lit("-"), date_format("data_da_coleta", "MM")))
    .groupBy("estado_sigla","produto")
    .pivot("ano_da_coleta")
    .agg(
        round(avg("valor_de_venda"), 3).alias("avg"),
        round(max("valor_de_venda"), 3).alias("max"),
        round(min("valor_de_venda"), 3).alias("min")
    )
    .orderBy("produto")
    )

preco_medio_max_min_estado_ano_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lakehouse.gold.preco_medio_max_min_estado_ano_df")

In [0]:
preco_medio_estado_mes = (
    spark.read.table("lakehouse.silver.precos_combustiveis")
    .withColumn("ano_mes_coleta", concat(year("data_da_coleta"), lit("-"), date_format("data_da_coleta", "MM")))
    .groupBy("ano_mes_coleta","ano_da_coleta","mes_da_coleta","estado_sigla","produto")
    .agg(
        round(avg("valor_de_venda"), 3).alias("preco_medio_venda"),
        round(max("valor_de_venda"), 3).alias("preco_maximo"),
        round(min("valor_de_venda"), 3).alias("preco_minimo"),
        round(stddev("valor_de_venda"), 3).alias("desvio_padrao"),
    )
    .orderBy("estado_sigla", "produto", "ano_mes_coleta")
    )

preco_medio_estado_mes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lakehouse.gold.preco_medio_estado_mes")

In [0]:

w = Window.partitionBy("estado_sigla", "produto").orderBy("ano_mes_coleta")

variacao_temporal_df = (
    spark.read.table("lakehouse.gold.preco_medio_estado_mes")
    .withColumn(
        "preco_mes_anterior",
        lag(col("preco_medio_venda")).over(w)
    )
    .withColumn(
        "variacao_pct_mensal",
        round(((col("preco_medio_venda") - col("preco_mes_anterior")) / col("preco_mes_anterior")) * 100, 2)
    )
    .withColumn(
        "media_movel_3m",
        round(avg(col("preco_medio_venda")).over(w.rowsBetween(-2, 0)), 3 )
    )
    .withColumn(
        "media_movel_12m",
        round(avg(col("preco_medio_venda")).over(w.rowsBetween(-11, 0)), 3 )
    )
)

variacao_temporal_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lakehouse.gold.variacao_temporal")

In [0]:
base_df = spark.read.table("lakehouse.gold.preco_medio_estado_mes")

media_anual_df = (
    base_df
    .groupBy("ano_da_coleta", "estado_sigla", "produto")
    .agg(
        avg("preco_medio_venda").alias("preco_ano")
    )
)

sazonalidade_mensal_df = (
    base_df.alias("m")
    .join(
        media_anual_df.alias("a"),
        on=["ano_da_coleta", "estado_sigla", "produto"],
        how="inner"
    )
    .select(
        col("m.ano_mes_coleta"),
        col("m.estado_sigla"),
        col("m.produto"),
        round(col("m.preco_medio_venda"), 4).alias("preco_mensal"),
        round((col("m.preco_medio_venda") / col("a.preco_ano")) * 100, 2).alias("indice_sazonalidade"),
        round(col("m.preco_medio_venda") - col("a.preco_ano"), 4).alias("desvio_da_media_anual")
    )
)

sazonalidade_mensal_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lakehouse.gold.sazonalidade_mensal")